<a href="https://colab.research.google.com/github/broadinstitute/missense-pfes/blob/main/Copy_of_Compute_PFES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
if 'google.colab' in sys.modules:
  %pip install g2papi
import g2papi


In [ ]:
# @title # Input your gene/protein (HGNC symbol/UniProt accession) and a variant (e.g. M1V)
# @markdown Forms support many types of fields.

Gene = 'UMOD'  # @param {type: "string"}
UniProt = 'P07911'  # @param {type: "string"}
Variant = 'C77Y'  # @param {type: "string"}

# Input your protein (UniProt Accession) and a variant (e.g. M1V)
 - Import protein features from Genomics 2 Proteins portal via g2papi

In [ ]:
# Get protein features as a pandas dataframe
protein_features = g2papi.get_protein_features(gene, uid)
protein_features.fillna('-', inplace=True)

protein_features

/tmp/ipykernel_155/1166826388.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '-' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  protein_features.fillna('-', inplace=True)


,residueId,AA,Amino acid residues,Amino acid properties,Secondary structure (PDBe/SIFTS),Secondary structure (DSSP 3-state)*,Secondary structure (DSSP 9-state)*,Accessible surface area (Å²)*,Phi angle (degrees)*,Psi angle (degrees)*,...,Intra-chain Non-bonded interaction (PDB),Intra-chain Non-bonded interaction (AlphaFold2),Intra-chain Disulfide bond (PDB),Intra-chain Disulfide bond (AlphaFold2),Intra-chain Salt bridge (PDB),Intra-chain Salt bridge (AlphaFold2),Inter-chain Hydrogen bond (PDB),Inter-chain Non-bonded interaction (PDB),Inter-chain Disulfide bond (PDB),Inter-chain Salt bridge (PDB)
0,1,M,Methionine,Aliphatic,-,C (loop/coil),C (loop/coil),254,360.0,119.5,...,-,-,-,-,-,-,-,-,-,-
1,2,G,Glycine,"Special, lack of a chiral carbon, smallest ami...",-,C (loop/coil),C (loop/coil),79,-150.8,169.3,...,-,-,-,-,-,-,-,-,-,-
2,3,Q,Glutamine,Polar/Neutral,-,C (loop/coil),C (loop/coil),190,-81.1,164.4,...,-,-,-,-,-,-,-,-,-,-
3,4,P,Proline,"Special, No backbone hydrogen",-,C (loop/coil),C (loop/coil),123,-91.6,172.7,...,-,-,-,-,-,-,-,-,-,-
4,5,S,Serine,Polar/Neutral,-,C (loop/coil),C (loop/coil),118,-151.9,135.2,...,-,-,-,-,-,-,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
635,636,T,Threonine,Polar/Neutral,-,C (loop/coil),C (loop/coil),133,-92.0,123.9,...,-,-,-,-,-,-,-,-,-,-
636,637,L,Leucine,Aliphatic,-,C (loop/coil),C (loop/coil),141,-119.7,102.6,...,-,-,-,-,-,-,-,-,-,-
637,638,T,Threonine,Polar/Neutral,-,C (loop/coil),C (loop/coil),130,-98.4,123.2,...,-,-,-,-,-,-,-,-,-,-
638,639,F,Phenylalanine,Aromatic,-,C (loop/coil),C (loop/coil),184,-143.9,129.5,...,-,-,-,-,-,-,-,-,-,-


In [ ]:
import requests
import pandas as pd
from io import StringIO

# Construct the public URL for the Google Cloud Storage object
bucket_name = 'g2p-portal'
file_path = 'portal_data/2026_q1_data/uniprot_metadata.tsv'
gcs_public_url = f'https://storage.googleapis.com/{bucket_name}/{file_path}'

try:
    response = requests.get(gcs_public_url)
    response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)

    file_content = response.text

    # Load the file_content into a pandas DataFrame
    df_uniprot_metadata = pd.read_csv(StringIO(file_content), sep='\t')

    # Extract UniProt ID and PANTHER_protein_class
    uniprot_panther_data = df_uniprot_metadata[['UniprotKB_Entry', 'PANTHER_protein_class']]

    # Filter the uniprot_panther_data DataFrame using the uid (assuming 'uid' is defined)
    panther_class_for_uid = uniprot_panther_data[uniprot_panther_data['UniprotKB_Entry'] == uid]['PANTHER_protein_class']

    # Check if a class was found and print it
    if not panther_class_for_uid.empty:
        print(f"PANTHER Protein Class for UniProt ID '{uid}':")
        display(panther_class_for_uid.iloc[0])
    else:
        print(f"No PANTHER Protein Class found for UniProt ID '{uid}'.")

except requests.exceptions.RequestException as e:
    print(f"Error accessing GCS file: {e}")
    print("This might be due to the file/bucket not being publicly accessible or the path being incorrect.")

PANTHER Protein Class for UniProt ID 'P07911':


'transmembrane signal receptor'

In [ ]:
import requests
import pandas as pd
from io import StringIO

# Raw URL for the CSV file on GitHub
github_csv_url = 'https://raw.githubusercontent.com/broadinstitute/missense-pfes/main/results/enrichment_OR_by_protein_class.csv'

try:
    response = requests.get(github_csv_url)
    response.raise_for_status() # Raise an exception for HTTP errors (4xx or 5xx)

    # Read the content into a pandas DataFrame
    enrichment_df = pd.read_csv(StringIO(response.text))

    print("Successfully loaded the enrichment data. Here's a preview:")
    display(enrichment_df.head())

    # --- Extract odd ratio for a given protein class ---
    # Replace 'YOUR_PROTEIN_CLASS_HERE' with the actual protein class you want to look up
    # Example protein classes might be 'enzymatic activity', 'receptor activity', etc.
    # You can find available protein classes by checking enrichment_df['Protein Class'].unique()

    desired_protein_class = 'YOUR_PROTEIN_CLASS_HERE' # e.g., 'DNA binding transcription factor activity'

    # Filter the DataFrame for the desired protein class
    odd_ratio_data = enrichment_df[enrichment_df['Protein Class'] == desired_protein_class]

    if not odd_ratio_data.empty:
        print(f"\nOdd Ratio for '{desired_protein_class}':")
        # Assuming 'Odd Ratio' is the column name for the odd ratio
        display(odd_ratio_data[['Protein Class', 'Odd Ratio']])
    else:
        print(f"\nNo data found for protein class: '{desired_protein_class}'.")
        print("Please check the exact spelling of the protein class.")

except requests.exceptions.RequestException as e:
    print(f"Error accessing the GitHub CSV file: {e}")
    print("Please ensure the URL is correct and the file is publicly accessible.")


Successfully loaded the enrichment data. Here's a preview:


,Unnamed: 0,All,All.1,All.2,All.3,All.4,All.5,All.6,DNA_metabolism_protein,DNA_metabolism_protein.1,...,transporter.4,transporter.5,transporter.6,unclassified,unclassified.1,unclassified.2,unclassified.3,unclassified.4,unclassified.5,unclassified.6
0,NaN,OR,CI_lo,CI_up,p_value,q_value,n_case_yes,n_ctrl_yes,OR,CI_lo,...,q_value,n_case_yes,n_ctrl_yes,OR,CI_lo,CI_up,p_value,q_value,n_case_yes,n_ctrl_yes
1,feature,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SS:B,2.2547679832146357,2.004584310445821,2.5361760199545387,1.0919405672706685e-42,1.653968800424689e-42,688.0,470.0,1.3268447629907152,0.6010317348381147,...,0.06514110802459713,81.0,40.0,2.1091895506183302,1.5819822275228492,2.8120926285016052,7.856937048887315e-07,1.123978494493602e-06,81.0,110.0
3,SS:E,2.090225163934419,2.0337138036126343,2.148306820843534,0.0,0.0,13002.0,10362.0,2.5653118212845176,2.1317058251685364,...,0.0017981628228759894,830.0,499.0,2.601244555563372,2.441886500345286,2.771002352849446,3.497656552096949e-184,2.001436804811032e-183,1971.0,2372.0
4,SS:G,1.4572690966166215,1.376189747991854,1.5431253016182145,1.174502825335709e-37,1.7281970144225434e-37,2331.0,2474.0,1.0260056568196103,0.7049609597419421,...,3.2331776860980716e-09,468.0,207.0,1.3527275791939217,1.173738242467372,1.5590119136488094,4.348249657446646e-05,5.669236895151956e-05,286.0,607.0


KeyError: 'Protein Class'